# Cleaning Downloaded Data from avian-influenza

Author: Alexander Maksiaev

Purpose: Clean downloaded data from Andersen Lab's avian-influenza GitHub, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"
* Before running this code, ensure that fork is updated

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "11-01-2021"
end_date = "02-20-2026"
date_range = start_date + "--" + end_date

# Maintenance genotypes
genotypes = ["B3.13", "D1.1", "D1.3"]

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"

references = home + "references/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
ncbi_complete = downloads + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/"

complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
combined_files = downloads + "Combinations/NCBI_Virus_Andersen/" + date_range + "_Antarctica_North_America_South_America/"
if not os.path.exists(combined_files): # checking if the directory exists or not
    os.makedirs(combined_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")

# # Get list of all genotypes
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])



## Read Metadata 

In [3]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]

print(len(metadata)) 
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_15056\3314208339.py:3: DtypeWarning: Columns (16,32,36) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")


20047
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
19676


## De-duplicate from NCBI Virus

In [4]:
# Get NCBI Virus SRA sequences
ncbi_sras = []
for dirpath, dirs, files in os.walk(ncbi_complete):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            # Some accessions may not be SRA
            for value in sra_accessions.values:
                if "SRR" in value:
                    ncbi_sras.append(value)
            # Some duplicates may only be such because of duplicate isolates
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                ncbi_sras.append(value)
            # Need partial isolates -- e.g. "012345-001" instead of "25-012345-001-original"
            partials = fasta_file["Partials"]
            for value in partials.values:
                ncbi_sras.append(value)
    break 

metadata["Partials"] = metadata["isolate"].apply(lambda x: partial_isolate(x) if x == x else x)

# Remove duplicates from Andersen
for value in ncbi_sras: # to remove
    if "SRR" in value:
        metadata = metadata[metadata["Run"] != value]
    
    metadata = metadata[metadata["isolate"] != value]
    metadata = metadata[metadata["Partials"] != value]
    # print(value)
    
print(len(metadata))

________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly__

## Naming convention -- relabeling sequences


>[SRA_Accession]|A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1, etc.

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

### Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata_unassigned = metadata[metadata["Genotype"].str.contains("Not assigned")]
metadata_genotypes = metadata[metadata["Genotype"].isin(genotypes)] # | metadata["Genotype"].str.contains("Not assigned")]

metadata = pd.concat([metadata_unassigned, metadata_genotypes])

print(len(metadata)) 
display(metadata)

9365


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,Partials,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
211,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,,010327-005,2025-05-09_10-47-27,SRR28834850.fa,Not assigned: Only 6 segments >98.0% match fou...,"NS:am1.1, MP:ea1, PB1:am4, NP:am8, NA:ea1, HA:ea1","am2.2:22-010445-001:PB2, am1.1:22-010085-001:N...","97.52%, 99.17%, 98.78%, 99.55%, 98.00%, 99.40%...","11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report
212,SRR28834851,WGS,146.19,29552271,PRJNA1102327,SAMN41100354,Viral,9992398,USDA-NVSL,2024-04-04,...,,010327-002,2025-05-09_10-47-27,SRR28834851.fa,Not assigned: No Matching Genotypes,"PB1:am4, PA:ea1, NA:ea1, HA:ea1, NS:am1.1, MP:...","am4:23-001855-001:PB1, ea1:22-003707-003:PA, e...","99.55%, 99.22%, 98.94%, 98.63%, 99.28%, 98.78%...","4, 2, 15, 16, 6, 12, 2, 9",Ran on FASTA - No Coverage Report
213,SRR28834852,WGS,145.66,20301625,PRJNA1102327,SAMN41100353,Viral,6839276,USDA-NVSL,2024-03-15,...,,010195-004,2025-05-09_10-47-27,SRR28834852.fa,Not assigned: Only 7 segments >98.0% match fou...,"HA:ea1, NA:ea1, NS:am1.1, PB1:am4, NP:am8, PB2...","ea1:22-003707-003:HA, ea1:22-003707-003:PA, ea...","98.83%, 92.66%, 99.15%, 99.28%, 99.55%, 99.33%...","20, 27, 12, 6, 4, 10, 0, 11",Ran on FASTA - No Coverage Report
215,SRR28834854,WGS,145.49,33095525,PRJNA1102327,SAMN41100351,Viral,10748503,USDA-NVSL,2024-02,...,,010071-004,2025-05-09_10-47-27,SRR28834854.fa,Not assigned: Only 5 segments >98.0% match fou...,"NP:am8, NA:ea1, PA:ea1, NS:am1.1, MP:ea1","am2.2:22-010445-001:PB2, am8:23-032005-001:NP,...","95.08%, 99.27%, 95.14%, 98.99%, 99.09%, 94.92%...","51, 11, 21, 14, 19, 22, 8, 11",Ran on FASTA - No Coverage Report
220,SRR28834887,WGS,223.14,64340257,PRJNA1102327,SAMN41106834,Viral,35609374,USDA-NVSL,2024-03-17,...,,009499-002,2025-05-09_10-45-53,SRR28834887.fa,Not assigned: No Matching Genotypes,"PB1:am4, NP:am8, NS:am1.1, NA:ea1, PA:ea1, HA:...","am4:23-001855-001:PB1, am8:23-032005-001:NP, a...","99.55%, 99.33%, 99.17%, 99.08%, 99.35%, 99.39%...","4, 10, 7, 13, 3, 4, 11, 3",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12296,SRR37041910,WGS,146.96,188159927,PRJNA980729,SAMN54969232,Viral,60342299,USDA-NVSL,2026-02,...,,000512-002,2026-01-30_05-32-33,SRR37041910.fa,D1.1,"PB1:ea3, HA:ea3, NA:am4N1, MP:ea3, NS:ea3, NP:...","ea3:22-013001-001:PB1, ea3:22-013001-001:HA, a...","99.21%, 99.35%, 98.85%, 99.59%, 98.93%, 99.47%...","18, 11, 13, 4, 9, 8, 21, 11",Ran on FASTA - No Coverage Report
12297,SRR37041911,WGS,147.32,163229981,PRJNA980729,SAMN54969231,Viral,52205426,USDA-NVSL,2026-02,...,,000512-001,2026-01-30_05-32-36,SRR37041911.fa,D1.1,"NS:ea3, MP:ea3, PB2:am24, NP:am13, NA:am4N1, P...","ea3:22-013001-001:NS, ea3:22-013001-001:MP, am...","98.81%, 99.59%, 99.47%, 99.47%, 98.76%, 99.21%...","10, 4, 12, 8, 14, 18, 11, 21",Ran on FASTA - No Coverage Report
12298,SRR37041912,WGS,147.14,196417352,PRJNA980729,SAMN54969230,Viral,62839301,USDA-NVSL,2026-02,...,,000708-001,2026-01-30_05-32-36,SRR37041912.fa,D1.1,"PB1:ea3, HA:ea3, NA:am4N1, NP:am13, PB2:am24, ...","ea3:22-013001-001:PB1, ea3:22-013001-001:HA, a...","99.16%, 99.18%, 98.85%, 99.53%, 99.56%, 99.21%...","19, 14, 13, 7, 10, 17, 2, 12",Ran on FASTA - No Coverage Report
12300,SRR37041914,WGS,144.02,214201962,PRJNA980729,SAMN54969228,Viral,68582911,USDA-NVSL,2026-02,...,,000705-005,2026-01-30_05-32-36,SRR37041914.fa,D1.1,"MP:ea3, HA:ea3, NP:am13, NS:ea3, PA:am4, NA:am...","ea3:22-013001-001:MP, ea3:22-013001-001:HA, am...","99.69%, 99.06%, 99.47%, 98.81%, 99.30%, 99.23%...","3, 16, 8, 10, 15, 8, 11, 17",Ran on FASTA - No Coverage Report


### Get specific geolocation

In [6]:


# Double-check state with genbank_mapping
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="left")
# Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x 
                                                                else x)
metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
metadata["Geo_Location"] = metadata["name_state_genbank"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x.split(" ")[-1]
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                        else 
                                                        x
                                                        )

# If USA-, delete -
metadata["Geo_Location"] = metadata["Geo_Location"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
# metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,seg,genbank_acc,genbank_seg,genbank_name,name_state_genbank,Geo_Location
0,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,"11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report,SRR28834850_HA_cns.fa,Consensus_SRR28834850_HA_cns_threshold_0.5_qua...,HA,PP824561.1,4.0,A/cattle/North Carolina/24-010327-005/2024,North Carolina,USA-NC
1,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,"11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report,SRR28834850_MP_cns.fa,Consensus_SRR28834850_MP_cns_threshold_0.5_qua...,MP,PP824562.1,7.0,A/cattle/North Carolina/24-010327-005/2024,North Carolina,USA-NC
2,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,"11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report,SRR28834850_NA_cns.fa,Consensus_SRR28834850_NA_cns_threshold_0.5_qua...,NaN,PP824563.1,6.0,A/cattle/North Carolina/24-010327-005/2024,North Carolina,USA-NC
3,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,"11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report,SRR28834850_NP_cns.fa,Consensus_SRR28834850_NP_cns_threshold_0.5_qua...,NP,PP824564.1,5.0,A/cattle/North Carolina/24-010327-005/2024,North Carolina,USA-NC
4,SRR28834850,WGS,144.82,83716380,PRJNA1102327,SAMN41100355,Viral,27974638,USDA-NVSL,2024-04-04,...,"11, 7, 12, 4, 27, 9, 15, 23",Ran on FASTA - No Coverage Report,SRR28834850_NS_cns.fa,Consensus_SRR28834850_NS_cns_threshold_0.5_qua...,NS,PP824565.1,8.0,A/cattle/North Carolina/24-010327-005/2024,North Carolina,USA-NC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20706,SRR37041910,WGS,146.96,188159927,PRJNA980729,SAMN54969232,Viral,60342299,USDA-NVSL,2026-02,...,"18, 11, 13, 4, 9, 8, 21, 11",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
20707,SRR37041911,WGS,147.32,163229981,PRJNA980729,SAMN54969231,Viral,52205426,USDA-NVSL,2026-02,...,"10, 4, 12, 8, 14, 18, 11, 21",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
20708,SRR37041912,WGS,147.14,196417352,PRJNA980729,SAMN54969230,Viral,62839301,USDA-NVSL,2026-02,...,"19, 14, 13, 7, 10, 17, 2, 12",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
20709,SRR37041914,WGS,144.02,214201962,PRJNA980729,SAMN54969228,Viral,68582911,USDA-NVSL,2026-02,...,"3, 16, 8, 10, 15, 8, 11, 17",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA


### Collection Dates

In [7]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(str(x), default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

0        2024-04-04
1        2024-04-04
2        2024-04-04
3        2024-04-04
4        2024-04-04
            ...    
20706       2026-02
20707       2026-02
20708       2026-02
20709       2026-02
20710       2026-02
Name: Collection_Date, Length: 20711, dtype: object


### Get host type

In [15]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
                wild_avian domestic_avian               cattle        feline  \
0         great_horned_owl       pheasant            dairy_cow           cat   
1             common_raven         turkey               cattle  domestic_cat   
2            cooper's_hawk        chicken  cattle milk product     feral_cat   
3             coopers_hawk          goose          bovine_milk        feline   
4                  peafowl    guinea_fowl              bovine   domestic-cat   
...                    ...            ...                  ...           ...   
1180                 finch            NaN                  NaN           NaN   
1181  glaucose winged gull            NaN                  NaN           NaN   
1182      chukar partridge            NaN                  NaN           NaN   
1183          embden goose            NaN                  NaN           NaN   
1184      pie billed grebe            NaN                  NaN           NaN   

       other_mammal       human     

In [16]:
# Ensure that user checks animal output
input("Check animals output. Afterwards, press ESCAPE to continue.")

''

In [17]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

### Make names using all the attributes we collected

In [18]:
# Make names

# metadata["isolate_name"] = metadata["genbank_name"]

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

metadata["isolate_name"] = np.where(metadata["genbank_name"] == "", "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)), metadata["genbank_name"])


names = ">" + metadata["Run"] + "|" + metadata["isolate_name"] + "|" + metadata["serotype"] + "|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x) if "-" not in str(x) else str(dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

# print(set(metadata["serotype"]))

5        >SRR28834850|A/cattle/North Carolina/24-010327...
13       >SRR28834851|A/cattle/North Carolina/24-010327...
21       >SRR28834852|A/cattle/New Mexico/24-010195-004...
22       >SRR28834854|A/cattle/United States/24-010071-...
30       >SRR28834887|A/cattle/Texas/24-009499-002/2024...
                               ...                        
20706    >SRR37041910|A/guineafowl/United States/26G005...
20707    >SRR37041911|A/goose/United States/26G00512-00...
20708    >SRR37041912|A/chicken/United States/26G00708-...
20709    >SRR37041914|A/chicken/United States/26G00705-...
20710    >SRR37041915|A/chicken/United States/26G00705-...
Name: Name, Length: 9233, dtype: object

In [19]:
# Drop duplicate runs 

metadata["Partials"] = metadata["isolate"].apply(partial_isolate)
metadata = metadata.drop_duplicates(subset=["Partials", "years"], keep="last") # Isolates may be identical

In [20]:
os.chdir(complete_files)

print(metadata)
# Save metadata
metadata.to_csv("Andersen_metadata_" + date_range + ".csv")

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
5      SRR28834850        WGS      144.82   83716380  PRJNA1102327   
13     SRR28834851        WGS      146.19   29552271  PRJNA1102327   
21     SRR28834852        WGS      145.66   20301625  PRJNA1102327   
22     SRR28834854        WGS      145.49   33095525  PRJNA1102327   
30     SRR28834887        WGS      223.14   64340257  PRJNA1102327   
...            ...        ...         ...        ...           ...   
20706  SRR37041910        WGS      146.96  188159927   PRJNA980729   
20707  SRR37041911        WGS      147.32  163229981   PRJNA980729   
20708  SRR37041912        WGS      147.14  196417352   PRJNA980729   
20709  SRR37041914        WGS      144.02  214201962   PRJNA980729   
20710  SRR37041915        WGS      147.84  151911160   PRJNA980729   

          BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
5      SAMN41100355          Viral  27974638   USDA-NVSL      2024-04-04  ... 

## Make FASTA files

In [21]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

# for segment in segments:
#     pair = "Unassigned_" + segment
#     pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]

                    # if "Not assigned" in genotype:
                    #     genotype = "Unassigned"
                    # print(header)
                    print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: No Matching Genotypes
Not assigned: Only 7 segments >98.0% match found

## Concatenate with NCBI Virus

In [22]:
# Create fasta files 

os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty

        output_path = complete_files + pair + "_" + date_range + ".fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR28752446|A/blackbird/Texas/24-008354-001/2024|H5N1|USA-TX|2024-03-16|wild_avian|B3.13
>SRR28752447|A/cattle/Texas/24-009108-005/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752448|A/cattle/Texas/24-009108-004/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752449|A/cattle/Texas/24-009108-003/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752450|A/cattle/Texas/24-009108-002/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752451|A/cattle/Texas/24-009108-001/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752452|A/cattle/Texas/24-009088-002/2024|H5N1|USA-TX|2024-03-13|cattle|B3.13
>SRR28752453|A/cattle/Texas/24-009088-001/2024|H5N1|USA-TX|2024-03-13|cattle|B3.13
>SRR28752454|A/cattle/Texas/24-009087-001/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752456|A/cattle/Texas/24-009028-019/2024|H5N1|USA-TX|2024-03-20|cattle|B3.13
>SRR28752457|A/chicken/Texas/24-007264-003/2024|H5N1|USA-TX|2024-03-07|domestic_avian|B3.13
>SRR28752458|A/cattle/Texas/24-009028-009/2024|H5N1|USA-TX|2024-03-20|c

In [23]:
# Concatenate metadata sheets

os.chdir(ncbi_complete)
ncbi_metadata = pd.read_csv("NCBI_Virus_" + date_range + "_metadata.csv")
ncbi_metadata_csv = ncbi_metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]


# Clean up column names
metadata = metadata.rename(columns={"Run":"Identifier",  "genbank_name":"GenBank_Title", "serotype":"Serotype", "Geo_Location":"Geo_Location_Abrv", "years":"Years"})
metadata["Isolate"] = metadata["isolate_name"].apply(lambda x: x.split("/")[-2])
metadata_csv = metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]

os.chdir(combined_files)
print(ncbi_metadata_csv.columns)
print(metadata_csv.columns)
combined_metadata = pd.concat([ncbi_metadata_csv, metadata_csv], ignore_index=True)
combined_metadata.to_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_15056\3483243775.py:4: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  ncbi_metadata = pd.read_csv("NCBI_Virus_" + date_range + "_metadata.csv")


Index(['Identifier', 'GenBank_Title', 'Host', 'Collection_Date', 'Isolate',
       'Serotype', 'Segment', 'Genotype', 'Host_Type', 'Years',
       'Geo_Location_Abrv', 'Name'],
      dtype='object')
Index(['Identifier', 'GenBank_Title', 'Host', 'Collection_Date', 'Isolate',
       'Serotype', 'Genotype', 'Host_Type', 'Years', 'Geo_Location_Abrv',
       'Name'],
      dtype='object')


In [24]:
# Concatenate with new NCBI Virus sequences

os.chdir(combined_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

# andersen = home + "Andersen/complete/" + date_range + "/"

# NCBI Virus files
filenames_ncbi = []
# for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
for dirpath, dirs, files in os.walk(ncbi_complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

# Andersen files
filenames_andersen = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_andersen.append(file_name)
    break 

print(filenames_andersen)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    a_file_name = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    print(a_file_name)
    for nv_file in filenames_ncbi:
        nv_file_name = nv_file.split("/")[-1].split("_")[0] + "_" + nv_file.split("/")[-1].split("_")[1]
        print(nv_file_name)
        if a_file_name == nv_file_name: # We have a common genotype
            common_genotypes.add(a_file_name)
            filenames = [a_file, nv_file]
            with open(combined_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
            infile.close()
            outfile.close()

# print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        print(partial_filename_a)
        for segment in segments:
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile2:
                with open(a_file) as infile2:
                    for line in infile2:
                        outfile2.write(line)
                    infile2.close()
                outfile2.close()

for a_file in filenames_ncbi:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            print(partial_filename_a)
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile3:
                with open(a_file) as infile3:
                    for line in infile3:
                        outfile3.write(line)
                    infile3.close()
                outfile3.close()

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/Andersen_metadata_11-01-2021--02-20-2026.csv', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_HA_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_MP_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_NA_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_NP_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_NS_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/11-01-2021--02-20-2026/B3.13_PA_11-01-2021--02-20-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/A